# extra_epochs.ipynb

1 extra fine-tuning epoch starting from `baseline_injected_focal`.

Train = `train_gera` + `train_lorugec` + `train_all` (15k) + `rare_punct_injection` (2k).
Validation stays pure: `val_gera` + `val_lorugec`.

Saves to `finetuned_injected2_focal`.

In [1]:
# env preparation
import os
import sys
import json
import warnings

import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, DataCollatorForTokenClassification

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print('We are in Colab!')
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT_DIR = '/content/drive/MyDrive/omg_diploma_2025/restore_punct'
    DATA_DIR = os.path.join(ROOT_DIR, 'data')
    MODEL_DIR = os.path.join(ROOT_DIR, 'models')
    !pip install -q transformers datasets seqeval evaluate accelerate torchcrf
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
else:
    print('We are running locally!')
    ROOT_DIR = os.getcwd()
    DATA_DIR = os.path.join(ROOT_DIR, 'data')
    MODEL_DIR = os.path.join(ROOT_DIR, 'models')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
warnings.filterwarnings('ignore')

MODEL_NAME = 'DeepPavlov/rubert-base-cased-sentence'
MAX_LEN = 512

BATCH_SIZE = 4
GRAD_ACCUM = 4
logging_steps = 50

FT_EPOCHS = 1
FT_LR = 5e-6

BASELINE_MODEL_DIR = os.path.join(MODEL_DIR, 'baseline_injected_focal')
FINETUNED_MODEL_DIR = os.path.join(MODEL_DIR, 'finetuned_injected2_focal')

print('BASELINE_MODEL_DIR:', BASELINE_MODEL_DIR)
print('FINETUNED_MODEL_DIR:', FINETUNED_MODEL_DIR)

# label map
with open(os.path.join(DATA_DIR, 'label_map.json'), 'r', encoding='utf-8') as f:
    ID_TO_LABEL_RAW = json.load(f)
    id2label = {int(k): v for k, v in ID_TO_LABEL_RAW.items()}
    label2id = {v: k for k, v in id2label.items()}

LABELS = list(id2label.values())
NUM_LABELS = len(LABELS)
print('Loaded', NUM_LABELS, 'labels')

# tokenizer + collator
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer)

def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LEN,
    )

    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for pred_seq, lab_seq in zip(predictions, labels):
        for pred_id, lab_id in zip(pred_seq, lab_seq):
            if lab_id == -100:
                continue
            true_predictions.append(LABELS[pred_id])
            true_labels.append(LABELS[lab_id])

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average='weighted',
        zero_division=0,
    )
    accuracy = accuracy_score(true_labels, true_predictions)

    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}

def get_model(load_from_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModelForTokenClassification.from_pretrained(
        load_from_path,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )
    return model.to(device)

def get_trainer(model_to_train, train_ds, val_ds, output_dir, learning_rate, num_epochs, eval_strategy='epoch'):
    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy=eval_strategy,
        save_strategy='no',
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        logging_steps=logging_steps,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=torch.cuda.is_available(),
        report_to='none',
        remove_unused_columns=False,
    )
    from custom_trainer import FocalLossTrainer
    return FocalLossTrainer(
        model=model_to_train,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

def save_model(model_to_save, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    model_to_save.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print('Model saved ->', save_dir)

# dataset mixing
REPLAY_SAMPLE_SIZE = 15_000
INJECTOR_SAMPLE_SIZE = 2_000

train_gera = load_dataset('json', data_files={'train': os.path.join(DATA_DIR, 'train_gera.json')})['train']
train_lorugec = load_dataset('json', data_files={'train': os.path.join(DATA_DIR, 'train_lorugec.json')})['train']

train_all = load_dataset('json', data_files={'train': os.path.join(DATA_DIR, 'train_all.json')})['train']
baseline_sample = train_all.shuffle(seed=42).select(range(REPLAY_SAMPLE_SIZE))

rare_injector = load_dataset('json', data_files={'train': os.path.join(DATA_DIR, 'rare_punct_injection.json')})['train']
rare_sample = rare_injector.shuffle(seed=42).select(range(INJECTOR_SAMPLE_SIZE))

domain_train = concatenate_datasets([train_gera, train_lorugec])
mixed_train = concatenate_datasets([domain_train, baseline_sample, rare_sample]).shuffle(seed=42)

val_gera = load_dataset('json', data_files={'validation': os.path.join(DATA_DIR, 'val_gera.json')})['validation']
val_lorugec = load_dataset('json', data_files={'validation': os.path.join(DATA_DIR, 'val_lorugec.json')})['validation']
combined_val = concatenate_datasets([val_gera, val_lorugec]).shuffle(seed=42)

print('mixed_train total:', len(mixed_train))
print('combined_val total:', len(combined_val))

# tokenize + align
tokenized_train = mixed_train.map(
    align_labels_with_tokens,
    batched=True,
    remove_columns=['tokens', 'ner_tags'],
)

tokenized_val = combined_val.map(
    align_labels_with_tokens,
    batched=True,
    remove_columns=['tokens', 'ner_tags'],
)

# train
model_ft = get_model(load_from_path=BASELINE_MODEL_DIR)
trainer_ft = get_trainer(
    model_ft,
    tokenized_train,
    tokenized_val,
    output_dir=os.path.join(MODEL_DIR, 'bert-extra-injected2-continue_temp'),
    learning_rate=FT_LR,
    num_epochs=FT_EPOCHS,
    eval_strategy='epoch',
)

trainer_ft.train()
save_model(model_ft, FINETUNED_MODEL_DIR)

del model_ft, trainer_ft
torch.cuda.empty_cache()
print('extra_epochs.ipynb completed')


We are running locally!
BASELINE_MODEL_DIR: /home/temari/god please no diploma/restore_punct/models/baseline_injected_focal
FINETUNED_MODEL_DIR: /home/temari/god please no diploma/restore_punct/models/finetuned_injected2_focal
Loaded 28 labels


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Map:   0%|          | 0/797 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.088688,0.036291,0.959857,0.960120,0.958899,0.960120


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved -> /home/temari/god please no diploma/restore_punct/models/finetuned_injected2_focal
extra_epochs.ipynb completed
